In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

import sys
sys.path.insert(0, r"C:\Users\Andrei\HonorsProject\pytspl")

from pytspl.scnn import (
    MaskedReconstructionTrainer,
    SimplicialBatch,
    SimplicialConvolution,
    SimplicialConvolution2,
    build_cochains,
    build_normalized_operators,
)


In [2]:
import sys, torch
print(sys.executable)
print(torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("torch cuda:", torch.version.cuda)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


c:\Users\Andrei\HonorsProject\pytspl-env\Scripts\python.exe
2.10.0+cu126
cuda available: True
torch cuda: 12.6
NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [3]:
from pytspl import load_dataset

dataset = "scnn_paper"
sc, _, flow = load_dataset(dataset=dataset, only_2d=False)

topdim = sc.max_dim
print("topdim:", topdim)
print("shape:", sc.shape)
print("flow type:", type(flow))


c:\Users\Andrei\HonorsProject\pytspl-env\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


Generating coordinates using spring layout.
Num. of 0-simplices: 352
Num. of 1-simplices: 1474
Num. of 2-simplices: 3285
Num. of 3-simplices: 5019
Num. of 4-simplices: 5559
Num. of 5-simplices: 4547
Num. of 6-simplices: 2732
Num. of 7-simplices: 1175
Num. of 8-simplices: 343
Num. of 9-simplices: 61
Num. of 10-simplices: 5
Shape: (352, 1474, 3285, 5019, 5559, 4547, 2732, 1175, 343, 61, 5)
Max Dimension: 10
Coordinates: 352
Flow: 0
topdim: 10
shape: (352, 1474, 3285, 5019, 5559, 4547, 2732, 1175, 343, 61, 5)
flow type: <class 'dict'>


In [4]:
topdim = 5
batch_size = 1
colors = 1

print("Using topdim =", topdim)
print("M_d:", [sc.shape[d] for d in range(topdim + 1)])


Using topdim = 5
M_d: [352, 1474, 3285, 5019, 5559, 4547]


In [5]:
class MySCNNDeprecated(nn.Module):
    def __init__(self, topdim: int, conv_kind: str, colors=1, num_filters=30, variance=0.01, K1=2, K2=3):
        super().__init__()
        self.topdim = topdim
        self.conv_kind = conv_kind

        if conv_kind not in {"simplicial_convolution", "simplicial_convolution2"}:
            raise ValueError("conv_kind must be 'simplicial_convolution' or 'simplicial_convolution2'")

        self.layers = nn.ModuleList()
        for _ in range(topdim + 1):
            per_dim_layers = nn.ModuleList()
            in_c = colors
            hidden_c = num_filters * colors
            channel_plan = [in_c, hidden_c, hidden_c, colors]
            for i in range(len(channel_plan) - 1):
                c_in = channel_plan[i]
                c_out = channel_plan[i + 1]
                if conv_kind == "simplicial_convolution":
                    per_dim_layers.append(
                        SimplicialConvolution(
                            K=K2,
                            C_in=c_in,
                            C_out=c_out,
                            variance=variance,
                            basis="power",
                        )
                    )
                else:
                    per_dim_layers.append(
                        SimplicialConvolution2(
                            K1=K1,
                            K2=K2,
                            C_in=c_in,
                            C_out=c_out,
                            variance=variance,
                            basis="power",
                        )
                    )
            self.layers.append(per_dim_layers)

        self.act = nn.LeakyReLU()

    def forward(self, batch: SimplicialBatch):
        ys = []

        Ls = None
        if self.conv_kind == "simplicial_convolution":
            Ls = batch.metadata.get("Ls")
            if Ls is None:
                Ls = [(batch.Lls[d] + batch.Lus[d]).coalesce() for d in range(self.topdim + 1)]
                batch.metadata["Ls"] = Ls

        for d in range(self.topdim + 1):
            x = batch.xs[d]
            for i, layer in enumerate(self.layers[d]):
                if self.conv_kind == "simplicial_convolution":
                    x = layer(Ls[d], x)
                else:
                    x = layer(batch.Lls[d], batch.Lus[d], x)

                if i < len(self.layers[d]) - 1:
                    x = self.act(x)
            ys.append(x)
        return ys


In [6]:
import time
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

BASE_SEED = 1337
NUM_EPOCHS = 1000
REALIZATIONS = 10
MISSING_PCTS = [0.1, 0.2, 0.3, 0.4, 0.5]
EPSILON = 0.05

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

print(f"Using device: {DEVICE}")

# Precompute static tensors once; they do not depend on missing percentage or realization.
BASE_XS_TARGET = build_cochains(sc, topdim, batch_size=batch_size, channels=colors, device=DEVICE)
BASE_LLS, BASE_LUS = build_normalized_operators(sc, topdim, half_interval=True, device=DEVICE)
BASE_LS = [(ll + lu).coalesce() for ll, lu in zip(BASE_LLS, BASE_LUS)]


def make_batch_for_missing(missing_pct: float, seed: int) -> SimplicialBatch:
    xs_target = [x.clone() for x in BASE_XS_TARGET]
    xs_input = [x.clone() for x in xs_target]
    b = SimplicialBatch(
        xs=xs_input,
        xs_target=xs_target,
        Lls=BASE_LLS,
        Lus=BASE_LUS,
        metadata={"topdim": topdim, "Ls": BASE_LS},
    )
    b.mask_cochains(missing_pct, seed=seed)
    return b


def all_accuracy_per_degree(model: nn.Module, batch: SimplicialBatch, epsilon: float) -> list[float]:
    with torch.inference_mode():
        ys_eval = model(batch)

    out = []
    for d in range(topdim + 1):
        pred = ys_eval[d][0, 0, :].detach().cpu().numpy()
        true = batch.xs_target[d][0, 0, :].detach().cpu().numpy()

        thresh = np.maximum(np.abs(epsilon * true), 1e-12)
        ok_all = np.abs(pred - true) <= thresh
        out.append(float(ok_all.mean()))
    return out


experiment_records = []

for missing_pct in MISSING_PCTS:
    for model_name, conv_kind in [
        ("SimplicialConvolution", "simplicial_convolution"),
        ("SimplicialConvolution2", "simplicial_convolution2"),
    ]:
        t0 = time.perf_counter()
        per_realization = []

        for r in range(REALIZATIONS):
            seed = BASE_SEED + int(1000 * missing_pct) + r
            torch.manual_seed(seed)
            np.random.seed(seed)
            if DEVICE.type == "cuda":
                torch.cuda.manual_seed_all(seed)

            run_batch = make_batch_for_missing(missing_pct, seed=seed)
            model = MySCNNDeprecated(topdim=topdim, conv_kind=conv_kind, colors=colors).to(DEVICE)
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            criterion = nn.L1Loss(reduction="sum")

            trainer = MaskedReconstructionTrainer(
                model=model,
                optimizer=optimizer,
                criterion=criterion,
                topdim=topdim,
                realizations=1,
            )
            trainer.fit(
                run_batch,
                num_epochs=NUM_EPOCHS,
                print_every=100,
                realizations=1,
                zero_grad_set_to_none=True,
                realization_now=r,
            )

            per_realization.append(all_accuracy_per_degree(model, run_batch, EPSILON))

        arr = np.array(per_realization, dtype=float)
        means = np.nanmean(arr, axis=0)
        stds = np.nanstd(arr, axis=0)

        rec = {
            "Missing": f"{int(missing_pct * 100)}%",
            "Model": model_name,
        }
        for d in range(topdim + 1):
            rec[f"d={d} (M={sc.shape[d]})"] = f"{means[d]:.2f} +/- {stds[d]:.2f}"

        experiment_records.append(rec)
        elapsed_min = (time.perf_counter() - t0) / 60.0
        print(f"Done: missing={missing_pct:.1f}, model={model_name}, elapsed={elapsed_min:.1f} min")

results_df = pd.DataFrame(experiment_records)
results_df


Using device: cuda
realization=0 epoch=0 loss=183698.0
realization=0 epoch=100 loss=35546.98828125
realization=0 epoch=200 loss=19112.83984375
realization=0 epoch=300 loss=4519.466796875
realization=0 epoch=400 loss=1938.773193359375
realization=0 epoch=500 loss=1107.4263916015625
realization=0 epoch=600 loss=1265.00146484375
realization=0 epoch=700 loss=673.1459350585938
realization=0 epoch=800 loss=696.4324951171875
realization=0 epoch=900 loss=739.7267456054688
realization=1 epoch=0 loss=184305.0
realization=1 epoch=100 loss=35013.01171875
realization=1 epoch=200 loss=19687.4765625
realization=1 epoch=300 loss=7437.619140625
realization=1 epoch=400 loss=1783.3125
realization=1 epoch=500 loss=1153.2821044921875
realization=1 epoch=600 loss=914.3466186523438
realization=1 epoch=700 loss=941.6759033203125
realization=1 epoch=800 loss=725.2975463867188
realization=1 epoch=900 loss=739.3262329101562
realization=2 epoch=0 loss=184595.0
realization=2 epoch=100 loss=34699.68359375
realizati

,Missing,Model,d=0 (M=352),d=1 (M=1474),d=2 (M=3285),d=3 (M=5019),d=4 (M=5559),d=5 (M=4547)
0,10%,SimplicialConvolution,0.91 +/- 0.00,0.91 +/- 0.00,0.91 +/- 0.00,0.91 +/- 0.00,0.92 +/- 0.00,0.92 +/- 0.00
1,10%,SimplicialConvolution2,0.91 +/- 0.00,0.91 +/- 0.00,0.91 +/- 0.00,0.92 +/- 0.00,0.92 +/- 0.00,0.92 +/- 0.00
2,20%,SimplicialConvolution,0.81 +/- 0.00,0.82 +/- 0.00,0.82 +/- 0.00,0.83 +/- 0.00,0.84 +/- 0.00,0.84 +/- 0.00
3,20%,SimplicialConvolution2,0.81 +/- 0.00,0.82 +/- 0.00,0.83 +/- 0.00,0.83 +/- 0.00,0.84 +/- 0.00,0.85 +/- 0.00
4,30%,SimplicialConvolution,0.72 +/- 0.00,0.73 +/- 0.00,0.74 +/- 0.00,0.75 +/- 0.00,0.76 +/- 0.00,0.76 +/- 0.00
5,30%,SimplicialConvolution2,0.72 +/- 0.00,0.73 +/- 0.00,0.74 +/- 0.00,0.75 +/- 0.00,0.76 +/- 0.00,0.77 +/- 0.00
6,40%,SimplicialConvolution,0.63 +/- 0.01,0.64 +/- 0.00,0.65 +/- 0.00,0.66 +/- 0.00,0.68 +/- 0.00,0.69 +/- 0.00
7,40%,SimplicialConvolution2,0.63 +/- 0.01,0.64 +/- 0.00,0.65 +/- 0.00,0.66 +/- 0.00,0.68 +/- 0.00,0.69 +/- 0.00
8,50%,SimplicialConvolution,0.53 +/- 0.01,0.55 +/- 0.01,0.56 +/- 0.00,0.58 +/- 0.00,0.59 +/- 0.00,0.61 +/- 0.00
9,50%,SimplicialConvolution2,0.53 +/- 0.01,0.55 +/- 0.01,0.56 +/- 0.00,0.58 +/- 0.00,0.59 +/- 0.00,0.61 +/- 0.00
